In [1]:
# ---------------------------------------------------------------------------------- #
# CAMIA: Context‑Aware Membership Inference Attack – **Debugged & Streamlined**       #
# ---------------------------------------------------------------------------------- #
#   • Cell‑0  –  Imports, device config, global knobs                                 #
#   • Cell‑1  –  Helper for canonical HuggingFace model names                         #
#   • Cell‑2  –  Core signal extraction (fixed signs, faster LZ‑diff)                 #
#   • Cell‑3  –  P‑value combiners + Group‑PCA / LogReg attack (clean & weighted)     #
#   • Cell‑4  –  ROC/PR metric helpers                                                #
#   • Cell‑5  –  Minimal demo / grid‑search driver                                    #
#   • Cell‑6  –  Optional diagnostics & per‑feature tables                            #
#                                                                                        
# **Bug‑fixes & changes vs original**                                                 #
#   ✓ removed duplicated / legacy cells                                               #
#   ✓ fixed sign of `slope_*`, `apen_*`, and `lz_bins*_rep{1,2}_diff`                 #
#   ✓ proper weighting of the neighbourhood p‑value in all combiners                  #
#   ✓ single global z‑score  *or* per‑group whitening (toggle via `USE_WHITENING`)    #
#   ✓ neighbour gamma applied on coefficients (no variance distortion)                #
#   ✓ memory‑safe CE batching, early‑exit cache, tqdm progress                        #
#   ✓ explicit seed control for full determinism                                      #
# ---------------------------------------------------------------------------------- #

# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell‑0 · imports & global config                                          ║
# ╚════════════════════════════════════════════════════════════════════════════╝
import os, sys, json, math, random, itertools, bisect, glob, textwrap
from itertools import islice
import numpy as np
import pandas as pd
import importlib
import torch, torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets  import load_dataset
from tqdm.auto import tqdm, trange
from sklearn.decomposition import PCA
from sklearn.linear_model  import LogisticRegression
from sklearn.metrics       import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from collections import defaultdict

# reproducibility -------------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# device ----------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✔ using {DEVICE}")

# master knobs – tweak here ----------------------------------------------------
class C:                      # simple namespace
    MAX_LEN_CE       = 1024   # LM loss truncation
    CE_SUB_BS        = 16     # mini‑batch for CE pass
    FP16_CE          = True
    # ‑‑ neighbour settings
    NEI_K            = 0      # neighbours per passage (0 ⇒ vanilla CAMIA)
    NEI_WEIGHT       = 9      # weight in p‑value combiner (Edgington et al.)
    NEI_GAMMA        = 1.0    # multiplicative boost inside LogReg attack
    # signal hyper‑params
    CUT_TPRIMES      = {"T":None,"200":200,"300":300}
    CB_TAU_LIST      = [1,2,3]
    LZ_BIN_LIST      = [3,4,5]
    SLOPE_TPRIMES    = [600,800,1000]
    APEN_TPRIMES     = [600,800,1000]
    STORE_FULL_SEQ   = False  # saves RAM, switch on for diagnostics only

✔ using cuda


In [2]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell‑1 · helper: (suite,size) → HF repo                                    ║
# ╚════════════════════════════════════════════════════════════════════════════╝

def get_model_name(suite: str, size: str, deduped: bool = False) -> str:
    suite, size = suite.lower(), size.lower()
    if suite == "pythia":
        base = f"EleutherAI/pythia-{size}"
        return base + ("-deduped" if deduped else "")
    if suite == "gpt-neo":
        mapping = {"125m":"125M","1.3b":"1.3B","2.7b":"2.7B"}
        if size not in mapping:
            raise ValueError(f"Unsupported GPT‑Neo size: {size}")
        return f"EleutherAI/gpt-neo-{mapping[size]}"
    raise ValueError(f"Unknown suite: {suite}")

# ── dynamic per-GPU ceiling ────────────────────────────────────────────────
def fit_batch_size(model) -> None:        # <- changed signature
    """
    Shrinks C.CE_SUB_BS until one forward pass fits the free VRAM.
    No impact on the numeric results (only speed).
    """
    bs = C.CE_SUB_BS
    while True:
        try:
            dummy = torch.ones(bs, C.MAX_LEN_CE, dtype=torch.long,
                               device=DEVICE)
            with torch.amp.autocast(device_type='cuda',
                                    dtype=torch.float16,
                                    enabled=C.FP16_CE):
                model(dummy, attention_mask=torch.ones_like(dummy))
            break
        except torch.cuda.OutOfMemoryError:
            bs //= 2
            assert bs >= 1, "even BS=1 does not fit – lower MAX_LEN_CE"
            torch.cuda.empty_cache()
    if bs != C.CE_SUB_BS:
        print(f"⚠️  Low-VRAM mode → CE_SUB_BS={bs}")
    C.CE_SUB_BS = bs


In [3]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell‑2 · Signal extraction (bug‑fixed)                                     ║
# ╚════════════════════════════════════════════════════════════════════════════╝
#  –  build 72 numeric signals per passage (CAMIA §3)                           

from contextlib import nullcontext
from torch import autocast as _amp

safe_mean = lambda arr: float(np.mean(arr)) if len(arr) else 0.0
_diff     = lambda a,b: (a-b) if (np.isfinite(a) and np.isfinite(b)) else 0.0

# --------------------------------------------------------------------------- #
#   *Neighbour loading helper* – One‑liner wrapper; caches stay external      #
# --------------------------------------------------------------------------- #

def load_cached_neighbours(domain, split, tag):
    if C.NEI_K == 0:
        return None
    path = f"cache/neigh_{domain}_{split}_{tag}.json"
    with open(path) as f:
        table = json.load(f)
    # pad / clip so every row has exactly K entries (important!)
    for i,row in enumerate(table):
        if len(row) < C.NEI_K:
            table[i] = row + row[:1]*(C.NEI_K-len(row))
        else:
            table[i] = row[:C.NEI_K]
    return table

# --------------------------------------------------------------------------- #
#   Core primitives: CE losses, sequence stats, LZ complexity, slope, ApEn    #
# --------------------------------------------------------------------------- #

# ─── 2. _ce_losses  (memory-streamed, numerically identical) ────────────────
@torch.no_grad()
def _ce_losses(texts, model, tok, slice_len: int = 256):
    if not texts:
        return []

    enc = tok(texts,
              return_tensors="pt",
              padding=True,
              truncation=True,
              max_length=C.MAX_LEN_CE
             ).to(model.device)

    with torch.amp.autocast(device_type='cuda',
                            dtype=torch.float16,
                            enabled=C.FP16_CE):
        logits = model(enc.input_ids,
                       attention_mask=enc.attention_mask).logits[:, :-1]

    tgt  = enc.input_ids[:, 1:]
    mask = enc.attention_mask[:, 1:]

    B, T, V = logits.shape
    losses  = []

    # ── stream over the time dimension in fp32 chunks ───────────────────────
    for s in range(0, T, slice_len):
        e = min(s + slice_len, T)
        slab   = logits[:, s:e].float()            # (B, S, V)  fp32
        tgt_sl = tgt[:,   s:e]

        logp = torch.log_softmax(slab, dim=-1)     # still (B, S, V)
        ce   = -logp.gather(-1, tgt_sl.unsqueeze(-1)).squeeze(-1)  # (B, S)
        losses.append(ce)

        del slab, logp, ce
        torch.cuda.empty_cache()

    ce_full = torch.cat(losses, dim=1)             # (B, T)

    return [ce_full[i, : int(mask[i].sum())].tolist() for i in range(B)]


# ------------------------------------------------------------- helper stats #

def _ce_stream(text_list, model, tok,
               stream_bs=C.CE_SUB_BS, max_len=C.MAX_LEN_CE):
    out = []
    for i in range(0, len(text_list), stream_bs):
        out.extend(_ce_losses(text_list[i:i+stream_bs], model, tok))
        torch.cuda.empty_cache()
    return out


def _count_below(seq, thr, Tprime):
    sub = seq if Tprime is None else seq[:Tprime]
    return sum(v<=thr for v in sub)/len(sub) if sub else 0.0

def _count_below_mean(seq, L=None):
    sl = seq if L is None else seq[:L]
    if not sl:
        return 0.0
    m = safe_mean(sl)
    return sum(v<=m for v in sl)/len(sl)

def _count_below_prev_mean(seq, L=None):
    sl = seq if L is None else seq[:L]
    if len(sl)<=1:
        return 0.0
    below,csum = 0, sl[0]
    for i,v in enumerate(sl[1:],1):
        below += v<=csum/i; csum+=v
    return below/(len(sl)-1)

# -------- LZ76 complexity ---------------------------------------------------

def _lz_complexity(seq, bins, Tprime):
    sub = seq if Tprime is None else seq[:Tprime]
    if not sub:
        return 0.0
    arr = np.asarray(sub); lo,hi = arr.min(), arr.max()
    if hi-lo < 1e-12:
        return 1.0
    edges   = np.linspace(lo, hi+1e-9, bins)
    symbols = ''.join(chr(65+np.digitize(v,edges)) for v in arr)
    i,c=0,1
    while i<len(symbols)-1:
        l=1
        while symbols[i:i+l] in symbols[:i] and i+l<len(symbols):
            l+=1
        i+=l; c+=1
    return float(c)

# -------- slope & ApEn ------------------------------------------------------

def _replicate_if_short(seq, desired):
    if not seq or len(seq)>=desired:
        return seq
    out=list(seq)
    while len(out)<desired:
        out.extend(seq[:desired-len(out)])
    return out

def _slope_signal(loss_seq, Tprime):
    if not loss_seq:
        return 0.0
    y = np.asarray(_replicate_if_short(loss_seq, Tprime)[:Tprime])
    x = np.arange(len(y))
    var = np.mean((x - x.mean())**2)
    if var==0:
        return 0.0
    cov = np.mean((x - x.mean())*(y - y.mean()))
    return float(cov/var)   # negative for members

def _approximate_entropy(seq, Tprime, m=8, r=0.8):
    seq = _replicate_if_short(seq, Tprime)[:Tprime]
    if len(seq)<m+2:
        return 0.0
    vals=np.asarray(seq)
    def _phi(mm):
        N=len(vals)-mm+1
        if N<=0:
            return 0.0
        patterns=np.array([vals[i:i+mm] for i in range(N)])
        C=np.sum(np.max(np.abs(patterns[:,None]-patterns[None,:]),axis=2)<=r,axis=0)/N
        return np.mean(np.log(C+1e-12))
    return float(_phi(m)-_phi(m+1))

# -------- token diversity ---------------------------------------------------

def token_diversity(text,tok):
    toks=tok.tokenize(text)
    return len(set(toks))/len(toks) if toks else 0.0

# --------------------------------------------------------------------------- #
#   Neighbour Δ helper (vectorised, memory‑safe)                              #
# --------------------------------------------------------------------------- #

@torch.no_grad()
def _mean_ce_many(texts, model, tok, bs=4):
    # returns np.ndarray shape (N,)
    txts=[str(t) for t in texts]
    losses=[]
    for i in range(0,len(txts),bs):
        losses.extend([safe_mean(s) for s in _ce_losses(txts[i:i+bs],model,tok)])
        torch.cuda.empty_cache()
    return np.asarray(losses,dtype=np.float32)

def add_nei_delta(base_dicts, neighbour_table, model, tok):
    if C.NEI_K == 0 or neighbour_table is None:
        return base_dicts

    neighbour_table = neighbour_table[:len(base_dicts)]
    flat = list(itertools.chain.from_iterable(row for row in neighbour_table))
    nei_ce = _mean_ce_many(flat, model, tok).reshape(len(base_dicts), C.NEI_K)

    out = []
    for d, nei in zip(base_dicts, nei_ce):
        base_ce = safe_mean(d.get("loss_seq_orig", [])) if "loss_seq_orig" in d else d["cut_T"]
        delta   = float(base_ce - nei.mean())
        if not np.isfinite(delta):            # <── new
            delta = 0.0                       #      guard
        nd = d.copy()
        nd["nei_delta"] = delta
        out.append(nd)
    return out


# --------------------------------------------------------------------------- #
#   gather_signals_for_batch – builds one dict per passage                     #
# --------------------------------------------------------------------------- #

@torch.no_grad()
def gather_signals_for_batch(texts, model, tok, neighbour_rows=None):
    max_len = C.MAX_LEN_CE

    # 1) 3× truncations × 3× repetitions
    variants={(t,r):[] for t in ("T","200","300") for r in (0,1,2)}

    def _reps(txt,tp):
        if tp is None:
            base=txt
        else:
            ids=tok(txt)["input_ids"][:tp]
            base=tok.decode(ids,skip_special_tokens=True)
        return base, f"{base} {base}", f"{base} {base} {base}"

    for t in texts:
        for tag,tp in (("T",None),("200",200),("300",300)):
            b,r1,r2=_reps(t,tp)
            variants[(tag,0)].append(b)
            variants[(tag,1)].append(r1)
            variants[(tag,2)].append(r2)

    # batched CE
    losses = {k: _ce_stream(v, model, tok) for k, v in variants.items()}

    # 2) build feature dicts
    feats=[]
    for i,orig in enumerate(texts):
        L=lambda tag,rp: losses[(tag,rp)][i]
        d={}

        # — ❶ cut / cal / ppl / calppl —
        for tag,tp in C.CUT_TPRIMES.items():
            if tp is None:
                txt_trim=orig
            else:
                ids_trim=tok(orig)["input_ids"][:tp]
                txt_trim=tok.decode(ids_trim,skip_special_tokens=True)
            div=max(token_diversity(txt_trim,tok),1e-12)
            lb,lr1,lr2 = L(tag,0),L(tag,1),L(tag,2)
            for fam,post in (
                ("cut", lambda x:x),
                ("cal", lambda x:x/div),
                ("ppl", lambda x:math.exp(x)),
                ("calppl", lambda x:math.exp(x)/div)):
                vb,vr1,vr2=map(post,map(safe_mean,(lb,lr1,lr2)))
                d[f"{fam}_{tag}"]=vb
                d[f"{fam}_{tag}_rep1_diff"]=_diff(vb,vr1)  
                d[f"{fam}_{tag}_rep2_diff"]=_diff(vb,vr2)

        # — ❷ count‑below CONST (invert) —
        for tau in C.CB_TAU_LIST:
            lb,lr1,lr2=L("200",0),L("200",1),L("200",2)
            b,r1,r2=(_count_below(x,tau,200) for x in (lb,lr1,lr2))
            b,r1,r2=(1-b,1-r1,1-r2)
            k=f"cb_const{tau}"
            d[k]=b; d[f"{k}_rep1_diff"]=_diff(b,r1); d[f"{k}_rep2_diff"]=_diff(b,r2)

        # — ❸ count‑below MEAN/prev‑mean (invert) —
        for tag in C.CUT_TPRIMES:
            lb,lr1,lr2=L(tag,0),L(tag,1),L(tag,2)
            base,r1,r2=(_count_below_mean(x) for x in (lb,lr1,lr2))
            base,r1,r2=(1-base,1-r1,1-r2)
            k=f"cbm_{tag}"
            d[k]=base 
            d[f"{k}_rep1_diff"]=_diff(base,r1)
            d[f"{k}_rep2_diff"]=_diff(base,r2)
            d[f"cbpm_{tag}"]=1-_count_below_prev_mean(lb)

        # — ❹ LZ, slope, ApEn —
        for bins in C.LZ_BIN_LIST:
            lb,lr1,lr2 = L("200",0),L("200",1),L("200",2)
            b,r1,r2    = (_lz_complexity(x,bins,200) for x in (lb,lr1,lr2))
            k=f"lz_bins{bins}"
            d[k]=b
            d[f"{k}_rep1_diff"]=_diff(r1,b)          
            d[f"{k}_rep2_diff"]=_diff(r2,b)

        # slope / ApEn (costly) – do it only when really needed
        if C.SLOPE_TPRIMES or C.APEN_TPRIMES or C.STORE_FULL_SEQ:
            lb_full = L("T",0)
            for tp in C.SLOPE_TPRIMES:
                d[f"slope_{tp}"] = -_slope_signal(lb_full, tp)
            for tp in C.APEN_TPRIMES:
                d[f"apen_{tp}"]  = -_approximate_entropy(lb_full, tp)
            if C.STORE_FULL_SEQ:
                d["loss_seq_orig"] = lb_full

        # Safeguard against non-finite values
        for k, v in list(d.items()):
            if isinstance(v, float) and not np.isfinite(v):
                d[k] = 0.0              # or: del d[k]

        feats.append(d)

    # neighbour delta (optional)
    if C.NEI_K and neighbour_rows is not None:
        feats = add_nei_delta(feats, neighbour_rows, model, tok)

    # ------------------------------------------------------------------
    # final sanity-check before we hand the dicts back
    for d in feats:
        for k, v in list(d.items()):
            # accept ints, Python floats, *and* NumPy scalar floats
            if isinstance(v, (int, float, np.floating)) and not np.isfinite(v):
                d[k] = 0.0          # or: del d[k]
    # ------------------------------------------------------------------

    return feats

def gather_signals_for_dataset(texts, model, tokenizer, *,
                               tag: str,
                               domain_name: str,
                               split_name: str,
                               batch_size: int = 32,
                               **kw):
    """
    Convenience wrapper: iterates over `texts` in manageable batches
    and attaches the correct neighbour rows (if NEI_K>0).
    All extra kwargs are forwarded to `gather_signals_for_batch`.
    """
    nei_tbl = load_cached_neighbours(domain_name, split_name, tag)
    out = []

    for s in tqdm(range(0, len(texts), batch_size),
                  desc=f"Signal-extraction ({tag})"):
        e = s + batch_size
        out.extend(
            gather_signals_for_batch(
                texts[s:e], model, tokenizer,
                neighbour_rows=None if nei_tbl is None else nei_tbl[s:e],
                **kw
            )
        )
        torch.cuda.empty_cache()

    return out


In [4]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell‑3 · Combiners & attack models                                        ║
# ╚════════════════════════════════════════════════════════════════════════════╝

CLIP=1e-5
USE_WHITENING=True   # toggle – if True we do per‑group whitening and **omit** global z‑score

# ---------- helper: group name ---------------------------------------------

def _group(k):
    if k.startswith("nei"):
        return "nei"
    for p in ("cut","calppl","ppl","cal","cb_const","cbm","cbpm","lz_bins","slope","apen"):
        if k.startswith(p):
            return p.split("_")[0]
    return "misc"

# ---------- p‑value handling (weighted) -------------------------------------

def _pvalue(x, ref_arr):
    n=ref_arr.size
    rank=np.searchsorted(ref_arr, x, side="right")
    return rank/(n+1)


def _combine_weighted(pvals, method="edgington"):
    """pvals is list[(p,w)]"""
    if not pvals:
        return 0.0
    clip=lambda p: min(max(p,CLIP),1-CLIP)
    if method=="edgington":
        return sum(w*p for p,w in pvals)
    if method=="fisher":
        return sum(-2*w*math.log(clip(p)) for p,w in pvals)
    if method=="pearson":
        return sum(-2*w*math.log(clip(1-p)) for p,w in pvals)
    if method=="george":
        return sum(w*math.log(clip(p)/(1-clip(p))) for p,w in pvals)
    raise ValueError(method)


def build_reference(nonmember_dicts):
    ref=defaultdict(list)
    for d in nonmember_dicts:
        for k,v in d.items():
            if isinstance(v,(int,float)) and np.isfinite(v):
                ref[k].append(v)
    for k,arr in ref.items():
        ref[k]=np.sort(np.asarray(arr,dtype=np.float32))
    return ref


def pv_score_batch(dicts, ref, method="edgington"):
    scores=[]
    for d in dicts:
        pvals=[]
        for k,v in d.items():
            if k not in ref or not np.isfinite(v):
                continue
            w = (C.NEI_WEIGHT if k=="nei_delta" else 1)
            pvals.append((_pvalue(v, ref[k]), w))
        scores.append(_combine_weighted(pvals, method))
    return scores

# ---------- Group-PCA + LogReg --------------------------------------------
PCA_DIM_PER_GROUP = 2          # unchanged

def logistic_fit(member_dicts, nonmember_dicts):
    Xd = member_dicts + nonmember_dicts
    y  = np.array([1]*len(member_dicts) + [0]*len(nonmember_dicts))

    # 1) raw matrix
    keys = sorted({k for d in Xd for k,v in d.items()
                   if isinstance(v,(int,float)) and np.isfinite(v)})
    raw  = np.array([[d.get(k,0.0) for k in keys] for d in Xd],
                    dtype=np.float32)

    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0, copy=False)
    
    # 2) centre & scale once – always
    mu, sigma = raw.mean(0), raw.std(0) + 1e-9
    raw_z     = (raw - mu) / sigma

    # 3) per-group PCA / whitening
    grp2cols, pcs_parts, pca_dict, nei_mask = defaultdict(list), [], {}, []
    for i,k in enumerate(keys):
        grp2cols[_group(k)].append(i)

    for g, cols in grp2cols.items():
        sub = raw_z[:, cols]
        if sub.shape[1] == 1:
            pcs = sub                        # centred, unit-var already
            pca_dict[g] = None
        else:
            p = PCA(n_components=min(PCA_DIM_PER_GROUP, sub.shape[1]),
                    whiten=USE_WHITENING)
            pcs = p.fit_transform(sub)
            pca_dict[g] = p
        pcs_parts.append(pcs)
        nei_mask.extend([g == "nei"] * pcs.shape[1])

    X = np.concatenate(pcs_parts, axis=1)

    # 4) neighbour γ – work on a *copy* so repeated calls are safe
    nei_mask = np.asarray(nei_mask, bool)
    Xg       = X.copy()
    Xg[:, nei_mask] *= C.NEI_GAMMA

    # 5) train LR ------------------------------------------------------------
    clf = LogisticRegression(max_iter=1000, solver="lbfgs").fit(Xg, y)

    # 6) pack metadata – freeze the γ that was used
    meta = dict(keys       = keys,
                grp2cols   = grp2cols,
                pca        = pca_dict,
                mu         = mu,
                sigma      = sigma,
                nei_mask   = nei_mask,
                nei_gamma  = C.NEI_GAMMA)
    return clf, meta

def logistic_score_batch(dicts, clf, meta):
    # 1) rebuild raw matrix -----------------------
    raw = np.array([[d.get(k,0.0) for k in meta["keys"]] for d in dicts],
                   dtype=np.float32)

    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0, copy=False)  # <── new
    
    raw_z = (raw - meta["mu"]) / meta["sigma"]

    # 2) project into per-group PC space ----------
    pcs_parts = []
    for g, cols in meta["grp2cols"].items():
        sub = raw_z[:, cols]
        p   = meta["pca"][g]
        pcs = sub if p is None else p.transform(sub)
        pcs_parts.append(pcs)

    Xq = np.concatenate(pcs_parts, axis=1)

    # 3) apply the *frozen* γ ---------------------
    Xq[:, meta["nei_mask"]] *= meta["nei_gamma"]

    # 4) return -P(member) ------------------------
    return -clf.predict_proba(Xq)[:, 1]


In [5]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell‑4 · ROC metric helper                                                ║
# ╚════════════════════════════════════════════════════════════════════════════╝

def auc_tpr01(pos_scores, neg_scores):
    """AUC and TPR at *exactly* 1 % FPR (linear interpolation)."""
    y = [1]*len(pos_scores) + [0]*len(neg_scores)
    s = -np.concatenate([pos_scores, neg_scores])
    fpr, tpr, _ = roc_curve(y, s)

    # interpolate TPR at FPR = 0.01
    if fpr[0] >= .01:
        tpr01 = tpr[0]
    else:
        i = np.searchsorted(fpr, 0.01)
        tpr01 = np.interp(0.01, fpr[i-1:i+1], tpr[i-1:i+1])

    return roc_auc_score(y, s), tpr01


In [6]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Cell-5 · Grid-search driver (saves CSV/JSON artefacts)                    ║
# ╚════════════════════════════════════════════════════════════════════════════╝
"""
Run a full CAMIA + Neighbour comparison sweep and write results exactly like
the original notebook.  Safe to execute as-is; adjust GRID_* lists for your
own ablation studies.
"""
import os, json, time, pandas as pd, numpy as np
from accelerate import Accelerator

# ---------------- experiment grid ----------------
GRID_NEI_K      = [0, 25, 100]
GRID_NEI_WEIGHT = [9, 18, 36, 72]          # matched 1-to-1 with GAMMA
GRID_NEI_GAMMA  = [1, 2, 3, 4]
N_RUNS          = 10
OUT_PARENT      = "new_results_may22"

# ---------------- static model / data -----------
domain_name  = "pile_cc"
split_name   = "ngram_7_0.2"
num_samples  = 1000
suite,size,dedup = "gpt-neo","1.3b",False
model_tag    = f"{suite}{size}{'-dedup' if dedup else ''}".lower()

print(f"⌚ Grid-search on {domain_name}  model {model_tag}")

# dataset
hf_token = os.getenv("HF_TOKEN", "")
ds = load_dataset("iamgroot42/mimir", domain_name,
                  split=split_name, token=hf_token)
num_samples = min(num_samples, len(ds["member"]))
all_members = ds["member"][:num_samples]
all_nonmem  = ds["nonmember"][:num_samples]

# model + tokenizer
model_name = get_model_name(suite,size,dedup)
print("⤵ loading", model_name)
tok   = AutoTokenizer.from_pretrained(model_name)
tok.pad_token = tok.eos_token
#tok.add_special_tokens({'pad_token':'[PAD]'})
acc   = Accelerator()
model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,     # bigger range, same speed
            low_cpu_mem_usage=True)
model = acc.prepare(model).eval()

fit_batch_size(model)

# one-off feature extraction at K=0 (re-used for all K)
print("◆ Extracting base CAMIA signals (K=0)…")
C.NEI_K = 0
base_mem = gather_signals_for_dataset(all_members, model, tok,
                                      tag="member",
                                      domain_name=domain_name,
                                      split_name=split_name)

base_non = gather_signals_for_dataset(all_nonmem, model, tok,
                                      tag="nonmember",
                                      domain_name=domain_name,
                                      split_name=split_name)

print(f"◆   done – {len(base_mem)} member + {len(base_non)} non-member")

PVAL_METHODS = ("edgington", "fisher", "pearson", "george")
best_metric  = {}

# ------------------------------------------------------------------ #
#  pre-compute the 10 calibration / test splits – one per run        #
# ------------------------------------------------------------------ #
splits = []
for run in range(N_RUNS):
    rng = np.random.default_rng(SEED + run)
    idx_mem = rng.permutation(num_samples)
    idx_non = rng.permutation(num_samples)
    cut = int(num_samples * 0.30)
    splits.append((
        idx_mem[:cut],   idx_mem[cut:],     # member cal , member test
        idx_non[:cut],   idx_non[cut:]      # non-mem cal, non-mem test
    ))

# ───────────────────────── grid loop ─────────────────────────────────────────
for K in GRID_NEI_K:
    C.NEI_K = K
    mem_feat = add_nei_delta(
        base_mem,
        load_cached_neighbours(domain_name, split_name, "member"),
        model, tok
    )
    non_feat = add_nei_delta(
        base_non,
        load_cached_neighbours(domain_name, split_name, "nonmember"),
        model, tok
    )

    Ws = GRID_NEI_WEIGHT if K else [0]
    Gs = GRID_NEI_GAMMA  if K else [1]
    for W, GAM in zip(Ws, Gs):
        C.NEI_WEIGHT = W
        C.NEI_GAMMA  = GAM
        out_dir = f"{OUT_PARENT}/{domain_name}_{model_tag}/K{K}_W{W}"
        os.makedirs(out_dir, exist_ok=True)
        print(f"◇ K={K:3}  W={W:2}  γ={GAM}")

        records = []
        for run in range(N_RUNS):
            run_tag = f"run_{run:02d}"
            run_dir = f"{out_dir}/{run_tag}"
            os.makedirs(run_dir, exist_ok=True)

            mcal, mtest, ncal, ntest = splits[run]         
            mem_cal   = [mem_feat[i] for i in mcal]
            mem_test  = [mem_feat[i] for i in mtest]
            non_cal   = [non_feat[i] for i in ncal]
            non_test  = [non_feat[i] for i in ntest]
            
            # ── p-value attacks ------------------------------------------------
            ref = build_reference(non_cal)
            pval_auc, pval_tpr = {}, {}
            for m in PVAL_METHODS:
                s_mem = pv_score_batch(mem_test, ref, method=m)
                s_non = pv_score_batch(non_test, ref, method=m)
                pval_auc[m], pval_tpr[m] = auc_tpr01(s_mem, s_non)

            # ── Logistic-PCA attack ------------------------------------------
            clf, meta   = logistic_fit(mem_cal, non_cal)
            s_mem_l     = logistic_score_batch(mem_test, clf, meta)
            s_non_l     = logistic_score_batch(non_test, clf, meta)
            auc_log, tpr_log = auc_tpr01(s_mem_l, s_non_l)

            # ── console progress line ----------------------------------------
            print(
                f"  run {run_tag}  "
                + "  ".join(f"AUC_{m[:3]} {pval_auc[m]:.3f}" for m in PVAL_METHODS)
                + f"  AUC_log {auc_log:.3f} | "
                f"TPR01_edg {pval_tpr['edgington']*100:5.2f}%  "
                f"TPR01_log {tpr_log*100:5.2f}%"
            )

            # ── build raw-feature CSV ---------------------------------------------------
            rows = []

            for j, i in enumerate(mtest):
                rows.append({"idx": int(i),
                            "membership": 1,
                            "set_split": "test",
                            **mem_test[j]})
            for j, i in enumerate(ntest):
                rows.append({"idx": int(i),
                            "membership": 0,
                            "set_split": "test",
                            **non_test[j]})

            pd.DataFrame(rows).to_csv(
                f"{run_dir}/results_{domain_name}_{model_tag}.csv",
                index=False
)

            # ── per-run summary JSON ----------------------------------------
            rec = dict(
                run       = run,  K=K,  W=W,  gamma=GAM,
                **{f"auc_{m}"   : float(pval_auc[m])  for m in PVAL_METHODS},
                **{f"tpr01_{m}" : float(pval_tpr[m])  for m in PVAL_METHODS},
                auc_log   = float(auc_log),
                tpr01_log = float(tpr_log),
            )
            json.dump(
                rec,
                open(f"{run_dir}/summary_{domain_name}_{model_tag}.json", "w"),
                indent=2
            )
            records.append(rec)

        # ── aggregate over the 10 runs ---------------------------------------
        df  = pd.DataFrame(records)
        agg = df.mean(numeric_only=True).to_dict()
        json.dump(agg, open(f"{out_dir}/aggregate.json", "w"), indent=2)

        # ── track best-of-grid -----------------------------------------------
        for m in (*[f"auc_{x}"   for x in PVAL_METHODS], "auc_log",
                  *[f"tpr01_{x}" for x in PVAL_METHODS], "tpr01_log"):
            if m not in best_metric or agg[m] > best_metric[m][0]:
                best_metric[m] = (agg[m], K, W, GAM)

# overall leaderboard
print("========= BEST over grid =========")
for m, (score, K, W, G) in best_metric.items():
    label = "γ" if "log" in m else "W"
    param = G if "log" in m else W
    print(f"{m:<18} {score:6.3f}   (K={K}, {label}={param})")
print("==================================")

with open(f"{OUT_PARENT}/{domain_name}_{model_tag}/grid_best.json", "w") as f:
    json.dump({m: {"score": s, "K": K, "W": W, "gamma": G}
               for m, (s, K, W, G) in best_metric.items()},
              f, indent=2)
print("✓ wrote overall leaderboard →",
      f"{OUT_PARENT}/{domain_name}_{model_tag}/grid_best.json")


⌚ Grid-search on pile_cc  model gpt-neo1.3b


Using the latest cached version of the module from /home/edgelab/.cache/huggingface/modules/datasets_modules/datasets/iamgroot42--mimir/99dd53483f73e1949849b6202ab5c394203fb920504c1307e17105565ac9e33b (last modified on Mon Apr  7 13:55:00 2025) since it couldn't be found locally at iamgroot42/mimir, or remotely on the Hugging Face Hub.


⤵ loading EleutherAI/gpt-neo-1.3B


2025-06-22 16:14:41.392452: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-22 16:14:41.412272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750608881.434899  148371 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750608881.441645  148371 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750608881.461084  148371 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

⚠️  Low-VRAM mode → CE_SUB_BS=1
◆ Extracting base CAMIA signals (K=0)…


Signal-extraction (member):   0%|          | 0/32 [00:00<?, ?it/s]

Signal-extraction (nonmember):   0%|          | 0/32 [00:00<?, ?it/s]

◆   done – 1000 member + 1000 non-member
◇ K=  0  W= 0  γ=1
  run run_00  AUC_edg 0.550  AUC_fis 0.453  AUC_pea 0.546  AUC_geo 0.550  AUC_log 0.555 | TPR01_edg  4.86%  TPR01_log  5.43%
  run run_01  AUC_edg 0.556  AUC_fis 0.447  AUC_pea 0.551  AUC_geo 0.555  AUC_log 0.535 | TPR01_edg  3.00%  TPR01_log  3.14%
  run run_02  AUC_edg 0.539  AUC_fis 0.463  AUC_pea 0.531  AUC_geo 0.537  AUC_log 0.542 | TPR01_edg  2.86%  TPR01_log  2.43%
  run run_03  AUC_edg 0.553  AUC_fis 0.449  AUC_pea 0.550  AUC_geo 0.553  AUC_log 0.558 | TPR01_edg  3.00%  TPR01_log  3.43%
  run run_04  AUC_edg 0.553  AUC_fis 0.450  AUC_pea 0.548  AUC_geo 0.553  AUC_log 0.539 | TPR01_edg  3.43%  TPR01_log  4.00%
  run run_05  AUC_edg 0.552  AUC_fis 0.450  AUC_pea 0.545  AUC_geo 0.551  AUC_log 0.536 | TPR01_edg  3.57%  TPR01_log  4.00%
  run run_06  AUC_edg 0.555  AUC_fis 0.448  AUC_pea 0.552  AUC_geo 0.554  AUC_log 0.552 | TPR01_edg  4.43%  TPR01_log  3.71%
  run run_07  AUC_edg 0.550  AUC_fis 0.455  AUC_pea 0.547  AUC_ge

In [ ]:
###############################################################################
# Cell 6 – Diagnostics for *one* saved run  (new folder layout)               #
###############################################################################
import os, json, ast, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import roc_auc_score, roc_curve

# --------------------------------------------------------------------------- #
# -------- pick WHAT you want to inspect ------------------------------------ #
RUN_DOMAIN     = "arxiv"      # ← one of your domains
RUN_MODEL_TAG  = "pythia70m-dedup"     # exactly as in Cell-6
RUN_K          = 0                    # neighbour count  (0 / 25 / 100 …)
RUN_W          = 0                    # neighbour weight (0 for K=0)
RUN_ID         = "run_00"              # run_00 … run_09
OUT_PARENT_D      = "new_results_may22"
# --------------------------------------------------------------------------- #

RUN_DIR = (f"{OUT_PARENT_D}/{RUN_DOMAIN}_{RUN_MODEL_TAG}/"
           f"K{RUN_K}_W{RUN_W}/{RUN_ID}")

csv_path  = f"{RUN_DIR}/results_{RUN_DOMAIN}_{RUN_MODEL_TAG}.csv"
json_path = f"{RUN_DIR}/summary_{RUN_DOMAIN}_{RUN_MODEL_TAG}.json"

if not os.path.exists(csv_path):
    raise FileNotFoundError(csv_path)

df   = pd.read_csv(csv_path)
meta = json.load(open(json_path))

# ----------- basic splits --------------------------------------------------- #
df_test = (df if "set_split" not in df.columns
           else df[df.set_split == "test"])
df_mem  = df_test[df_test.membership == 1]
df_non  = df_test[df_test.membership == 0]

def tpr01(pos, neg):
    y   = [1]*len(pos) + [0]*len(neg)
    s   = -np.concatenate([pos, neg])
    fpr, tpr, _ = roc_curve(y, s)
    return tpr[np.argmin(np.abs(fpr-0.01))] * 100

# ----------- 1) histograms for every scalar feature ------------------------- #
skip = {"text_index", "membership", "set_split",
        "edgington_score", "logistic_score", "bestpv_score",
        "loss_seq_orig", "nei_texts"}

for feat in [c for c in df_test.columns if c not in skip]:
    plt.figure(figsize=(4,2))
    plt.hist(df_mem[feat], 40, alpha=.5, label="mem")
    plt.hist(df_non[feat], 40, alpha=.5, label="non")
    plt.title(f"{feat}  • TPR@1% = {tpr01(df_mem[feat], df_non[feat]):.1f}%")
    plt.legend(); plt.tight_layout(); plt.show()

# ----------- 2) token-loss curves (first / last 100) ------------------------ #
if "loss_seq_orig" in df_test.columns:
    def _avg_curve(dfs, first=True, K=100):
        seqs = [ast.literal_eval(s) if isinstance(s,str) else s
                for s in dfs.loss_seq_orig]
        arr  = np.full((len(seqs), K), np.nan)
        for i,s in enumerate(seqs):
            seg = s[:K] if first else s[-K:]
            if first:   arr[i,:len(seg)]  = seg
            else:       arr[i,-len(seg):] = seg
        return np.nanmean(arr,0)

    for lbl,f in (("first",True),("last",False)):
        plt.figure()
        plt.plot(_avg_curve(df_mem,f), label="mem")
        plt.plot(_avg_curve(df_non,f), label="non")
        plt.title(f"Average loss – {lbl} 100 tokens"); plt.legend(); plt.show()

# ----------- 3) ROC curves for attack-level scores -------------------------- #
for col in ("edgington_score","bestpv_score","logistic_score"):
    if col not in df_test.columns: continue
    y   = df_test.membership.values
    s   = -df_test[col].values
    auc = roc_auc_score(y, s)
    fpr,tpr,_ = roc_curve(y,s)
    plt.figure(); plt.plot(fpr, tpr, label=f"AUC {auc:.3f}")
    plt.title(col); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.show()

print("Diagnostics done – folder:", RUN_DIR)


In [7]:
###############################################################################
# Cell-8 – Detailed feature table  +  cross-dataset best-of grid             #
###############################################################################
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# ─────────── 1) CONFIG – pick ONE dataset / (K,W) for the feature table ─────
MODEL_TAG  = "gpt-neo1.3b"          # exactly as in Cell-5
DOMAIN     = "dm_mathematics"                    # one of your datasets
SELECT_K   = 0                          # K you want to inspect
SELECT_W   = 0                          # W you want to inspect
ROOT       = "new_results_may22"
# ─────────────────────────────────────────────────────────────────────────────

RUN_GLOB = (f"{ROOT}/{DOMAIN}_{MODEL_TAG}/K{SELECT_K}_W{SELECT_W}/"
            f"run_*/results_{DOMAIN}_{MODEL_TAG}.csv")

# --------------------------------------------------------------------------- #
def auc_tpr(arr_mem: np.ndarray, arr_non: np.ndarray):
    """Return (AUC , TPR@1 %FPR) for 1-D numeric arrays (scores: smaller → member)."""
    y   = np.r_[np.ones(len(arr_mem)), np.zeros(len(arr_non))]
    s   = -np.r_[arr_mem, arr_non]                      # note the minus!
    fpr, tpr, _ = roc_curve(y, s)
    # linear interpolation to exactly FPR = 0.01
    i  = np.searchsorted(fpr, 0.01)
    t1 = tpr[0] if i == 0 else np.interp(0.01, fpr[i-1:i+1], tpr[i-1:i+1])
    return float(roc_auc_score(y, s)), float(t1*100)

# columns we always ignore
FIXED_SKIP = {"idx", "set_split", "membership", "membership",
              "edgington_score", "logistic_score", "bestpv_score",
              "loss_seq_orig", "nei_texts"}

# ==================== PART A – per-feature table =========================== #
rows = []
for csv_path in glob.glob(RUN_GLOB):
    df = pd.read_csv(csv_path)                # rows = test set only
    mem = df[df.membership == 1]
    non = df[df.membership == 0]

    run_id = os.path.basename(os.path.dirname(csv_path))   # run_00, …

    for feat in df.columns:
        if feat in FIXED_SKIP:
            continue
        if not np.issubdtype(df[feat].dtype, np.number):
            continue                                  # skip non-numeric
        auc, tpr = auc_tpr(mem[feat].values, non[feat].values)
        rows.append(dict(feature=feat, run=run_id, AUC=auc, TPR01=tpr))

if not rows:
    raise RuntimeError(f"No CSV files found for DOMAIN={DOMAIN}  K={SELECT_K} W={SELECT_W}")

df_feat = pd.DataFrame(rows)

tbl_feat = (df_feat
            .groupby("feature")[["AUC", "TPR01"]]
            .mean()
            .round({"AUC": 3, "TPR01": 2})
            .T)
tbl_feat.index.name = "metric"

print(f"\n===  PER-FEATURE metrics — {DOMAIN}   (K={SELECT_K}, W={SELECT_W}) ===")
display(tbl_feat)

out_dir = f"{ROOT}/{DOMAIN}_{MODEL_TAG}"
tbl_feat.to_csv(f"{out_dir}/feature_metrics_K{SELECT_K}_W{SELECT_W}.csv")
print("saved per-feature table →", out_dir)

# ================ PART B – best-of-grid overview for every dataset ========= #
metrics = ["auc_edgington","auc_fisher","auc_pearson","auc_george","auc_log",
           "tpr01_edgington","tpr01_fisher","tpr01_pearson",
           "tpr01_george","tpr01_log"]

rows_best = []
for gb_path in glob.glob(f"{ROOT}/*_{MODEL_TAG}/grid_best.json"):
    domain = os.path.basename(os.path.dirname(gb_path)).rsplit("_", 1)[0]
    best   = json.load(open(gb_path))

    row = {"domain": domain}
    for m in metrics:
        if m in best:
            d = best[m]
            if m.endswith("_log"):
                row[m] = (round(d["score"], 3 if "auc" in m else 2),
                          d["K"],
                          f"γ={d['gamma']}" if d["K"] else "-")
            else:
                row[m] = (round(d["score"], 3 if "auc" in m else 2),
                          d["K"], d["W"])
        else:
            row[m] = np.nan
    rows_best.append(row)

df_best = (pd.DataFrame(rows_best)
           .set_index("domain")
           .sort_index())

print(f"\n===  BEST-OF-GRID per domain — model {MODEL_TAG} ===")
display(df_best)

out_csv = f"{ROOT}/overall_best_{MODEL_TAG}.csv"
df_best.to_csv(out_csv)
print("saved overall-best table →", out_csv)



===  PER-FEATURE metrics — dm_mathematics   (K=0, W=0) ===


feature,apen_1000,apen_600,apen_800,cal_200,cal_200_rep1_diff,cal_200_rep2_diff,cal_300,cal_300_rep1_diff,cal_300_rep2_diff,cal_T,...,ppl_200_rep2_diff,ppl_300,ppl_300_rep1_diff,ppl_300_rep2_diff,ppl_T,ppl_T_rep1_diff,ppl_T_rep2_diff,slope_1000,slope_600,slope_800
metric,,,,,,,,,,,,,,,,,,,,,
AUC,0.597,0.60,0.595,0.918,0.687,0.697,0.89,0.744,0.751,0.905,...,0.444,0.94,0.47,0.504,0.934,0.53,0.463,0.515,0.531,0.654
TPR01,0.320,1.27,0.320,47.460,2.860,3.970,40.32,3.330,4.290,50.000,...,0.950,65.87,0.00,0.000,63.650,0.00,0.000,0.000,2.060,1.270


saved per-feature table → new_results_may22/dm_mathematics_gpt-neo1.3b

===  BEST-OF-GRID per domain — model gpt-neo1.3b ===


,auc_edgington,auc_fisher,auc_pearson,auc_george,auc_log,tpr01_edgington,tpr01_fisher,tpr01_pearson,tpr01_george,tpr01_log
domain,,,,,,,,,,
arxiv,"(0.814, 0, 0)","(0.211, 25, 72)","(0.8, 100, 9)","(0.815, 100, 9)","(0.811, 0, -)","(0.3, 0, 0)","(0.0, 0, 0)","(0.24, 100, 9)","(0.31, 25, 9)","(0.32, 0, -)"
dm_mathematics,"(0.867, 0, 0)","(0.155, 100, 72)","(0.8, 0, 0)","(0.925, 0, 0)","(0.92, 0, -)","(0.21, 0, 0)","(0.0, 0, 0)","(0.1, 0, 0)","(0.61, 0, 0)","(0.43, 100, γ=1)"
github,"(0.894, 100, 9)","(0.13, 25, 72)","(0.887, 100, 9)","(0.896, 0, 0)","(0.893, 0, -)","(0.6, 100, 9)","(0.0, 0, 0)","(0.5, 100, 9)","(0.64, 25, 18)","(0.64, 0, -)"
hackernews,"(0.593, 0, 0)","(0.438, 100, 72)","(0.588, 100, 9)","(0.592, 0, 0)","(0.56, 100, γ=3)","(0.04, 0, 0)","(0.01, 25, 72)","(0.05, 0, 0)","(0.04, 0, 0)","(0.03, 0, -)"
pile_cc,"(0.551, 25, 9)","(0.455, 100, 72)","(0.548, 100, 18)","(0.551, 25, 18)","(0.542, 0, -)","(0.05, 25, 36)","(0.01, 25, 72)","(0.04, 100, 72)","(0.06, 25, 9)","(0.03, 0, -)"
pubmed_central,"(0.815, 100, 9)","(0.221, 25, 72)","(0.802, 100, 9)","(0.815, 0, 0)","(0.821, 0, -)","(0.23, 0, 0)","(0.0, 25, 72)","(0.23, 100, 9)","(0.26, 0, 0)","(0.21, 0, -)"


saved overall-best table → new_results_may22/overall_best_gpt-neo1.3b.csv
